In [ ]:
from transformers import RobertaTokenizer, RobertaForTokenClassification, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import numpy as np

# 1. Prepare dataset
dataset = load_dataset("conll2003")  # Replace with your dataset path

# 2. Define label mappings for your resume entities
label_list = ["O", "B-AGE", "I-AGE", "B-GENDER", "I-GENDER", "B-ADDRESS", "I-ADDRESS", 
              "B-SKILL", "I-SKILL", "B-EDUCATION", "I-EDUCATION", 
              "B-COURSE", "I-COURSE",  "B-EXPERIENCE", "I-EXPERIENCE", "B-CERTIFICATION",
               "I-CERTIFICATION"]

# 3. Create label-to-id mapping
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# 4. Load tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaForTokenClassification.from_pretrained(
    "roberta-base", 
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# 5. Tokenize and align labels
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    labels = []
    
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
            
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 6. Process the dataset
tokenized_dataset = dataset.map(
    tokenize_and_align_labels, 
    batched=True,
    remove_columns=dataset["train"].column_names
)

# 7. Define training arguments
training_args = TrainingArguments(
    output_dir="./roberta-resume-ner",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# 8. Define metrics for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    # Use seqeval for entity-level metrics
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 9. Initialize Trainer and train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

# 10. Save the model
model.save_pretrained("./roberta-resume-ner-final")
tokenizer.save_pretrained("./roberta-resume-ner-final")

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from seqeval.metrics import precision_score, recall_score, f1_score

# 1. Load custom dataset from a file
def load_data(file_path):
    sentences = []
    tags = []
    sentence = []
    tag = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                word, t = line.split()
                sentence.append(word)
                tag.append(t)
            else:
                if sentence:
                    sentences.append(sentence)
                    tags.append(tag)
                    sentence = []
                    tag = []
    if sentence:  # Catches the last sentence
        sentences.append(sentence)
        tags.append(tag)
    return sentences, tags

# Load training and evaluation data
train_sentences, train_tags = load_data("sample-annotate.txt")  # Training data (annotated)
eval_sentences, eval_tags = load_data("resumes_raw_text.txt")   # Evaluation data (raw text)

# 2. Define label mappings for your resume entities
label_list = [
    "O", "B-AGE", "I-AGE", "B-GENDER", "I-GENDER", "B-ADDRESS", "I-ADDRESS",
    "B-SOFT_SKILLS", "I-SOFT_SKILLS", "B-HARD_SKILLS", "I-HARD_SKILLS",
    "B-EDUCATION", "I-EDUCATION", "B-COURSE", "I-COURSE",
    "B-EXPERIENCE", "I-EXPERIENCE", "B-CERTIFICATION", "I-CERTIFICATION"
]
label_map = {label: i for i, label in enumerate(label_list)}

# 3. Load tokenizer and model with custom classification head
model_name = "xlm-roberta-large-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure the model for token classification with a new classification head
config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list))
model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)

# 4. Tokenization and dataset preparation
class ResumeDataset(Dataset):
    def __init__(self, sentences, tags, tokenizer, label_map, max_length=128):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        labels = self.tags[idx]

        # Tokenize words
        encoding = self.tokenizer(words,
                                  is_split_into_words=True,
                                  truncation=True,
                                  padding="max_length",
                                  max_length=self.max_length,
                                  return_tensors="pt")

        # Align labels with tokenized inputs
        word_ids = encoding.word_ids()
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Ignore tokens like [CLS], [SEP]
            elif word_idx != prev_word_idx:
                label_ids.append(self.label_map[labels[word_idx]])
            else:
                label_ids.append(self.label_map[labels[word_idx]])  # Inside word pieces
            prev_word_idx = word_idx

        encoding["labels"] = torch.tensor(label_ids, dtype=torch.long)
        return {key: val.squeeze(0) for key, val in encoding.items()}

# Create datasets
train_dataset = ResumeDataset(train_sentences, train_tags, tokenizer, label_map)
eval_dataset = ResumeDataset(eval_sentences, eval_tags, tokenizer, label_map)

# 5. Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

# 6. Compute evaluation metrics
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [[label_list[p] for (p, l) in zip(pred, label) if l != -100]
                        for pred, label in zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(pred, label) if l != -100]
                   for pred, label in zip(predictions, labels)]

    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)

    return {"precision": precision, "recall": recall, "f1": f1}

# 7. Initialize Trainer and start fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Save the fine-tuned model
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

print("Training completed and model saved!")


Fine-Tuning RoBERTa NER Model - March 3, 2025

In [6]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoConfig,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
from seqeval.metrics import precision_score, recall_score, f1_score

# 1. Load custom dataset from a file
def load_data(file_path):
    sentences = []
    tags = []
    sentence = []
    tag = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("-DOCSTART-"):  # Skip DOCSTART lines
                parts = line.split()
                if len(parts) >= 2:  # Ensure proper format
                    word, label = parts[-2], parts[-1]
                    sentence.append(word)
                    tag.append(label)
            else:
                if sentence:
                    sentences.append(sentence)
                    tags.append(tag)
                    sentence = []
                    tag = []
    if sentence:  # Catch the last sentence
        sentences.append(sentence)
        tags.append(tag)
    return sentences, tags

# Load training and evaluation data
train_sentences, train_tags = load_data("C:/Users/Acer/Desktop/Talaba,Ephraim/ARSwithPredictiveAnalytics/conll2003-traindata.conll")  # Change this to your dataset file
eval_sentences, eval_tags = load_data("C:/Users/Acer/Desktop/Talaba,Ephraim/ARSwithPredictiveAnalytics/conll-2003-evaldata.conll")  # Change this to your dataset file

In [7]:
# 2. Define label mappings for your custom entities
label_list = [
    "O", "B-B-SKILLS", "I-B-SKILLS", "B-I-SKILLS", "I-I-SKILLS",
    "B-B-CERTIFICATE", "I-B-CERTIFICATE", "B-I-CERTIFICATE", "I-I-CERTIFICATE",
    "B-B-ADDRESS", "I-B-ADDRESS", "B-I-ADDRESS", "I-I-ADDRESS",
    "B-B-WORK", "I-B-WORK", "B-I-WORK", "I-I-WORK"
]
label_map = {label: i for i, label in enumerate(label_list)}

In [ ]:
# 3. Load tokenizer and model with custom classification head
model_name = "xlm-roberta-large-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Configure the model for token classification with a new classification head
config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list))
model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)

In [ ]:
# 4. Tokenization and dataset preparation
class ResumeDataset(Dataset):
    def __init__(self, sentences, tags, tokenizer, label_map, max_length=128):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        labels = self.tags[idx]

        # Tokenize words
        encoding = self.tokenizer(words,
                                  is_split_into_words=True,
                                  truncation=True,
                                  padding="max_length",
                                  max_length=self.max_length,
                                  return_tensors="pt")

        # Align labels with tokenized inputs
        word_ids = encoding.word_ids()
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # Ignore tokens like [CLS], [SEP]
            elif word_idx != prev_word_idx:
                label_ids.append(self.label_map[labels[word_idx]])
            else:
                label_ids.append(self.label_map[labels[word_idx]])  # Inside word pieces
            prev_word_idx = word_idx

        encoding["labels"] = torch.tensor(label_ids, dtype=torch.long)
        return {key: val.squeeze(0) for key, val in encoding.items()}

# Create datasets
train_dataset = ResumeDataset(train_sentences, train_tags, tokenizer, label_map)
eval_dataset = ResumeDataset(eval_sentences, eval_tags, tokenizer, label_map)


In [ ]:
# 5. Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
# 6. Compute evaluation metrics
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [[label_list[p] for (p, l) in zip(pred, label) if l != -100]
                        for pred, label in zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(pred, label) if l != -100]
                   for pred, label in zip(predictions, labels)]

    precision = precision_score(true_labels, true_predictions)
    recall = recall_score(true_labels, true_predictions)
    f1 = f1_score(true_labels, true_predictions)

    return {"precision": precision, "recall": recall, "f1": f1}

In [ ]:
# 7. Initialize Trainer and start fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Save the fine-tuned model
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

print("Training completed and model saved!")